In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [3]:
movies_ml = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
movies_tmdb = pd.read_csv("tmdb_5000_movies.csv")

print("MovieLens Movies: ", movies_ml.shape)
print("Ratings: ", ratings.shape)
print("TMDB: ", movies_tmdb.shape)

MovieLens Movies:  (9742, 3)
Ratings:  (100836, 4)
TMDB:  (4803, 20)


In [5]:
movies_ml['title'] = movies_ml['title'].str.replace(r"\(\d{4}\)", "", regex = True)
movies_ml['title'] = movies_ml['title'].str.strip()

In [6]:
movies_tmdb['title'] = movies_tmdb['title'].str.strip()
movies_tmdb['oveerview'] = movies_tmdb['overview'].fillna('')

In [7]:
movies = pd.merge(movies_ml, movies_tmdb, on = 'title')
print("Merged Data:", movies.shape)
movies.head()

Merged Data: (2786, 23)


,movieId,title,genres_x,budget,genres_y,homepage,id,keywords,original_language,original_title,...,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,oveerview
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,30000000,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 35, ""...",http://toystory.disney.com/toy-story,862,"[{""id"": 931, ""name"": ""jealousy""}, {""id"": 4290,...",en,Toy Story,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-10-30,373554033,81.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,7.7,5269,"Led by Woody, Andy's toys live happily in his ..."
1,10,GoldenEye,Action|Adventure|Thriller,58000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 28, ""...",http://www.mgm.com/view/movie/757/Goldeneye/,710,"[{""id"": 701, ""name"": ""cuba""}, {""id"": 769, ""nam...",en,GoldenEye,...,"[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",1995-11-16,352194034,130.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,No limits. No fears. No substitutes.,6.6,1174,James Bond must unmask the mysterious head of ...
2,14,Nixon,Drama,44000000,"[{""id"": 36, ""name"": ""History""}, {""id"": 18, ""na...",NaN,10858,"[{""id"": 840, ""name"": ""usa president""}, {""id"": ...",en,Nixon,...,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1995-12-22,13681765,192.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Triumphant in Victory, Bitter in Defeat. He Ch...",7.1,71,An all-star cast powers this epic look at Amer...
3,15,Cutthroat Island,Action|Adventure|Romance,98000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",NaN,1408,"[{""id"": 911, ""name"": ""exotic island""}, {""id"": ...",en,Cutthroat Island,...,"[{""iso_3166_1"": ""FR"", ""name"": ""France""}, {""iso...",1995-12-22,10017322,119.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,The Course Has Been Set. There Is No Turning B...,5.7,136,"Morgan Adams and her slave, William Shaw, are ..."
4,16,Casino,Crime|Drama,52000000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 80, ""name...",NaN,524,"[{""id"": 383, ""name"": ""poker""}, {""id"": 726, ""na...",en,Casino,...,"[{""iso_3166_1"": ""FR"", ""name"": ""France""}, {""iso...",1995-11-22,116112375,178.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,No one stays at the top forever.,7.8,1307,The life of the gambling paradise – Las Vegas ...


In [10]:
movies['genres'] = movies['genres_x']
movies['genres'] = movies['genres'].fillna('')
movies['overview'] = movies['overview'].fillna('')
movies['content'] = movies['overview'] + " " + movies['genres']

In [11]:
tfidf = TfidfVectorizer(stop_words = 'english')
tfidf_matrix = tfidf.fit_transform(movies['content'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
movies = movies.reset_index()
indices = pd.Series(movies.index, index = movies['title']).drop_duplicates()

In [12]:
#Content Recommendation Function
def content_recommend(title, num = 5):
    if title not in indices:
        return "Movie not found"
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)[1:num+1]
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices]

In [13]:
#Collaborative Filtering
user_movie_matrix = ratings.pivot_table(index = 'userId', columns = 'movieId', values = 'rating').fillna(0)
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index = user_movie_matrix.index, columns = user_movie_matrix.index)

In [14]:
#Collaborative Recommendation
def collaborative_recommend(user_id, num = 5):
    similar_users = user_similarity_df[user_id].sort_values(ascending = False)[1:6]
    similar_movies = user_movie_matrix.loc[similar_users.index].mean().sort_values(ascending = False)
    return similar_movies.head(num)

In [24]:
#Hybrid Recommendation System
def hybrid_recommend(user_id, title, num = 5):
    content_rec = content_recommend(title, num)
    if isinstance(content_rec, str):
        return content_rec
    hybrid_scores = []
    similar_users = user_similarity_df[user_id].sort_values(ascending = False)[1:6]
    for movie in content_rec:
        movie_row = movies[movies['title'] == movie]
        if movie_row.empty:
            continue
        movie_id = movie_row['movieId'].values[0]
        ratings_list = user_movie_matrix.loc[similar_users.index][movie_id]
        ratings_list = ratings_list[ratings_list > 0]
        if len(ratings_list) > 0:
            score = ratings_list.mean()
        else: 
            score = ratings[ratings['movieId'] == movie_id]['rating'].mean()
        if pd.isna(score):
            continue
        hybrid_scores.append((movie, score))
    if len(hybrid_scores) == 0:
        return "No recommendation found"
    hybrid_scores = sorted(hybrid_scores, key = lambda x: x[1], reverse = True)
    return pd.DataFrame(hybrid_scores, columns = ['Movie', 'Predicted Rating'])

In [25]:
print("Content-Based:")
print(content_recommend("Toy Story"))
print("\n Collaborative: ")
print(collaborative_recommend(user_id = 1))
print("\n Hybrid: ")
print(hybrid_recommend(user_id = 1, title = "Toy Story"))

Content-Based:
2120               Toy Story 3
666                Toy Story 2
678            Man on the Moon
1771              Factory Girl
1737    For Your Consideration
Name: title, dtype: object

 Collaborative: 
movieId
1198    4.8
1200    4.8
2571    4.8
2028    4.7
296     4.5
dtype: float64

 Hybrid: 
                    Movie  Predicted Rating
0             Toy Story 3          4.109091
1             Toy Story 2          3.666667
2            Factory Girl          3.500000
3         Man on the Moon          3.487179
4  For Your Consideration          2.375000
